# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library with explicit referencing by `@id`. All record sets, fields, and columns are referred to by their unique `@id` values as defined by the Croissant schema.

### Dataset Source
The dataset is defined via a Croissant schema accessible here: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

#### Key topics covered:
- Data loading with metadata access
- Record set and field exploration (using `@id`)
- Data extraction and DataFrame creation
- Exploratory data analysis (filtering, normalization, grouping)
- Visualization of dataset columns and relationships


In [ ]:
# Install the `mlcroissant` library if not already present
!pip install mlcroissant --quiet

## 1. Data Loading

Load the metadata and available records from the dataset using `mlcroissant`. We start by setting the Croissant schema URL and loading the dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL (FAIR^2 dataset)
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset from the Croissant schema URL
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"Dataset: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}")

## 2. Data Overview

Next, we examine the structure of the dataset by listing all available record sets (`@id`) and, for each, their fields' `@id` and column `@id` if present.

Refer to the Croissant schema documentation for entity organization. Here, we enumerate top-level record sets, their fields, and columns, always by `@id`.

In [ ]:
# Enumerate all RecordSets in the dataset by their @id
print('Record Sets (@id):')
record_sets = []
for rs in dataset.record_sets:
    print(f"  - {rs['@id']}")
    record_sets.append(rs['@id'])
    
    # List fields for each record set
    if 'field' in rs:
        print("    Fields (@id):")
        if isinstance(rs['field'], list):
            for field in rs['field']:
                print(f"      - {field['@id']}")
                # Show columns (@id) if available
                if 'column' in field:
                    if isinstance(field['column'], list):
                        for col in field['column']:
                            print(f"        Column: {col['@id']}")
                    else:
                        print(f"        Column: {field['column']['@id']}")
        else:
            print(f"      - {rs['field']['@id']}")
            if 'column' in rs['field']:
                if isinstance(rs['field']['column'], list):
                    for col in rs['field']['column']:
                        print(f"        Column: {col['@id']}")
                else:
                    print(f"        Column: {rs['field']['column']['@id']}")

## 3. Data Extraction

Now, extract data from the record sets. As required, all entities (record sets and fields) are referenced by their `@id`.

- Define the list of record set `@id`s (from the overview section)
- Load each record set's data into a pandas DataFrame
- Display the columns (`@id`) and the first rows of a chosen record set

In [ ]:
# List record set @ids discovered in the overview (replace values with those appearing above if needed)
# If you cannot discover record sets, check dataset.record_sets for available options.
if not record_sets:
    print('No record sets found in the metadata.')
else:
    dataframes = {}
    for rs_id in record_sets:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set {rs_id} with {len(df)} records and columns: {list(df.columns)}")
        except Exception as e:
            print(f"Could not load record set {rs_id}: {e}")

    # Pick the first record set for example if any loaded
    if dataframes:
        main_record_set_id = list(dataframes.keys())[0]
        print(f"\nFirst five rows from record set {main_record_set_id}:")
        display(dataframes[main_record_set_id].head())
    else:
        main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Now, let's process a numeric field from the main record set:

- Filter records based on a threshold for a numeric field
- Normalize the numeric field
- Perform a grouping by a categorical field if available

For clarity, field selection is performed by `@id`.


In [ ]:
# --- User must select appropriate field @id's from record set ---
# For demonstration, pick the first numeric-looking field and a groupable field.

import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to find a numeric field from the DataFrame's columns by dtype or name
    numeric_field = None
    group_field = None
    for col in df.columns:
        # Heuristic for numeric field selection (float, int or column name)
        if df[col].dtype in [np.float64, np.int64, np.float32, np.int32]:
            numeric_field = col
            break
    # Fallback if no numeric field found, try string matching
    if numeric_field is None:
        for col in df.columns:
            if 'log_likelihood' in col.lower() or 'coefficient' in col.lower():
                numeric_field = col
                break

    # Find a potentially groupable/categorical field (heuristic)
    for col in df.columns:
        if df[col].dtype == object and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            if group_field != numeric_field:
                break

    if numeric_field is None:
        print('No numeric field found to analyze.')
    else:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        try:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records in '{main_record_set_id}' with '{numeric_field}' > {threshold:.3f} (using @id):")
            display(filtered_df.head())

            # Normalization
            norm_col = f"{numeric_field}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"\nNormalized '{numeric_field}' for filtered records (using @id):")
            display(filtered_df[[numeric_field, norm_col]].head())

            # Grouping by the group_field (if available)
            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].mean().reset_index()
                print(f"\nGrouped data by '{group_field}' (using @id):")
                display(grouped_df.head())
            else:
                print('No suitable group field found for grouping.')
        except Exception as e:
            print(f"Error during EDA: {e}")
else:
    print('No main record set loaded. Please check previous steps.')

## 5. Visualization

Let's visualize the numeric field and its relationships, if available, referencing all axes by their `@id`. Use pandas and matplotlib for plotting.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id and numeric_field:
    fig, axs = plt.subplots(1, 2, figsize=(13,5))
    # Histogram
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=axs[0], color='skyblue')
    axs[0].set_title(f"Distribution of '{numeric_field}' (@id)")
    axs[0].set_xlabel(numeric_field)

    # Boxplot by group field if possible
    if group_field and group_field in df.columns:
        top_groups = df[group_field].value_counts().index[:5]
        plot_data = df[df[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field, data=plot_data, ax=axs[1])
        axs[1].set_title(f"'{numeric_field}' by '{group_field}' (@id)")
        axs[1].set_xlabel(group_field)
        axs[1].set_ylabel(numeric_field)
    else:
        axs[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion

We demonstrated the use of the `mlcroissant` library for structured exploration of a FAIR^2-compliant dataset, referencing all entities by their `@id` as defined in the Croissant schema. 
- Loaded detailed metadata for the rangeland management practices dataset.
- Enumerated record sets and their available fields using `@id`.
- Loaded data for available record sets directly by `@id` and performed EDA, normalization, and grouping.
- Visualized numeric distributions, highlighting the power of referencing data entities by their schema-unique identifiers for reproducibility and clarity in scientific workflows.

> Continue adapting this workflow as new record sets or fields become available in the FAIR^2 schema.
